In [ ]:
'''1. Object Detection with YOLOv5 (Using Pretrained Weights)
Problem: Detect objects using pre-trained YOLOv5 model.'''

In [ ]:
import torch

# Load pretrained YOLOv5 model
model = torch.hub.load(
    'ultralytics/yolov5',
    'yolov5s',
    pretrained=True
)

# Run inference
results = model('image.jpg')

# Display results
results.show()

In [ ]:
'''2. Preprocessing Image for ResNet Input
Problem: Write a function to preprocess image for pretrained ResNet.'''

In [ ]:
import numpy as np
from tensorflow.keras.utils import load_img, img_to_array
from tensorflow.keras.applications.resnet50 import preprocess_input

def preprocess_image(image_path):
    # Load and resize image
    image = load_img(image_path, target_size=(224, 224))

    # Convert image to NumPy array
    image = img_to_array(image)

    # Add batch dimension
    image = np.expand_dims(image, axis=0)

    # ResNet50 preprocessing
    image = preprocess_input(image)

    return image

In [ ]:
'''3. CNN for CIFAR-10 Classification
Problem: Build a CNN model to classify CIFAR-10 images.'''

In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import Flatten, Dense, Dropout

# 1. Load CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

print("Training images:", x_train.shape)
print("Test images:", x_test.shape)

# 2. Normalize pixel values
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# 3. Build CNN
model = Sequential([
    
    # Convolutional block 1
    Conv2D(32, (3, 3), activation="relu", input_shape=(32, 32, 3)),
    MaxPooling2D((2, 2)),

    # Convolutional block 2
    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),

    # Convolutional block 3
    Conv2D(128, (3, 3), activation="relu"),

    # Convert feature maps into a vector
    Flatten(),

    # Fully connected layer
    Dense(128, activation="relu"),
    Dropout(0.5),

    # 10 CIFAR-10 classes
    Dense(10, activation="softmax")
])

# 4. Display model architecture
model.summary()

# 5. Compile model
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# 6. Train
history = model.fit(
    x_train,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1
)

# 7. Evaluate
test_loss, test_accuracy = model.evaluate(x_test, y_test)

print("Test Accuracy:", test_accuracy)

In [ ]:
'''4. Face Mask Detection – Real-Time Webcam Classifier
Problem: Build a real-time face mask detector using a trained CNN model and OpenCV.'''

In [ ]:
import cv2
import numpy as np
import tensorflow as tf

# -----------------------------
# Load trained CNN model
# -----------------------------
model = tf.keras.models.load_model("face_mask_model.keras")

# -----------------------------
# Load OpenCV face detector
# -----------------------------
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

# -----------------------------
# Start webcam
# -----------------------------
cap = cv2.VideoCapture(0)

while True:

    # Capture frame
    ret, frame = cap.read()

    if not ret:
        print("Could not access webcam")
        break

    # Convert frame to grayscale for face detection
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detect faces
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(50, 50)
    )

    # Process every detected face
    for (x, y, w, h) in faces:

        # Crop face
        face = frame[y:y+h, x:x+w]

        # Resize according to CNN input size
        face = cv2.resize(face, (224, 224))

        # Convert BGR → RGB
        face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)

        # Convert to float
        face = face.astype("float32") / 255.0

        # Add batch dimension
        face = np.expand_dims(face, axis=0)

        # -----------------------------
        # Make prediction
        # -----------------------------
        prediction = model.predict(face, verbose=0)[0][0]

        # Assuming:
        # 0 = With Mask
        # 1 = Without Mask

        if prediction < 0.5:
            label = "MASK"
            confidence = (1 - prediction) * 100
        else:
            label = "NO MASK"
            confidence = prediction * 100

        # -----------------------------
        # Draw bounding box
        # -----------------------------
        cv2.rectangle(
            frame,
            (x, y),
            (x+w, y+h),
            (0, 255, 0),
            2
        )

        # -----------------------------
        # Display label
        # -----------------------------
        text = f"{label}: {confidence:.1f}%"

        cv2.putText(
            frame,
            text,
            (x, y - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 0),
            2
        )

    # Display webcam
    cv2.imshow("Real-Time Face Mask Detection", frame)

    # Press Q to quit
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

# -----------------------------
# Release resources
# -----------------------------
cap.release()
cv2.destroyAllWindows()

In [ ]:
'''5. Custom Transfer Learning with Frozen + Trainable Layers
Problem: Load a pretrained MobileNetV2 model, freeze base layers, add custom classifier, and fine-tune the top few layers.
'''

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

IMG_SIZE = 224

# --------------------------------
# 1. Load pretrained MobileNetV2
# --------------------------------

base_model = tf.keras.applications.MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# --------------------------------
# 2. Freeze base model
# --------------------------------

base_model.trainable = False

# --------------------------------
# 3. Add custom classifier
# --------------------------------

model = models.Sequential([
    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation="relu"),

    layers.Dropout(0.3),

    layers.Dense(1, activation="sigmoid")
])

# --------------------------------
# 4. Compile
# --------------------------------

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# --------------------------------
# 5. Train classifier
# --------------------------------

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=5
)

# --------------------------------
# 6. Unfreeze MobileNetV2
# --------------------------------

base_model.trainable = True

# Freeze all except last 20 layers

for layer in base_model.layers[:-20]:
    layer.trainable = False

# --------------------------------
# 7. Recompile with low LR
# --------------------------------

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# --------------------------------
# 8. Fine-tune
# --------------------------------

history_fine = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=5
)

# --------------------------------
# 9. Save model
# --------------------------------

model.save("mobilenetv2_transfer_learning.keras")